# Lifecycle + RFM — chỉ dùng sự hiện diện, bỏ cờ hoạt động
**Lifecycle:** có snapshot → Win-back / New / Active theo ưu tiên; vắng 1–5 tháng → At-risk; vắng ≥6 tháng → Churned proxy. Không có Inactive hoặc Unknown trong trạng thái lifecycle.

**Active = hiện diện trong dataset**, không khẳng định đang giao dịch. Thiếu ngày mở quan hệ thì chưa xác nhận New; khách có snapshot vẫn là Active hoặc Win-back.

Mặc định toàn bộ khách, xuất từng tháng. Đánh giá đúng +1…+6; không cộng dồn. Ghép RFM theo customer_id + tháng, có bucket gần nhất kèm tuổi thông tin cho khách vắng.

**Cell 2:** INPUT_DIR = output notebook 01; RFM_ASSIGNMENT_DIR = assignments notebook 03. METADATA_PATH chỉ bổ sung fecha_alta để nhận diện New, không cần cờ hoạt động. **Cell 6** gán và xuất; **8** counts/transitions; **10** đánh giá trạng thái; **14/16** profile và benchmark RFM+lifecycle. Chạy lại toàn bộ để tránh dùng output rule cũ.


In [ ]:
from pathlib import Path
import json,re,uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
import pyarrow.parquet as pq
INPUT_DIR='/kaggle/input/notebooks/thientrantdt/01-r-f-m-analysis'
INPUT_FORMAT='rfm' # rfm hoặc legacy
METADATA_PATH='/kaggle/input/competitions/santander-product-recommendation/train_ver2.csv.zip' # optional: train_ver2.csv hoặc metadata customer_id, record_date/month_idx, fecha_alta; chỉ phục vụ New
RFM_ASSIGNMENT_DIR=None # folder assignments Hybrid đã chọn; optional
RFM_METHOD='Hybrid_REDA_F3_M5'
RFM_EVAL_FIT_MONTH='2015-11' # fit historical group rates on this source month, evaluate later sources
GROUP_RATE_PRIOR_N=100 # regularize response estimates; does not change R/F/M or groups
SAMPLE_MOD=1 # toàn bộ; 10 = thử ~10%, giữ lịch sử khách
CHURN_GAP=6; NEW_MONTHS=2
RULE_VERSION='lifecycle-v3-presence-gap6'
MIN_N=100; CSV_CHUNKSIZE=500_000
OUTPUT_DIR=Path('/kaggle/working/lifecycle_full' if Path('/kaggle/working').exists() else './lifecycle_full')
PARTITION_DIR=OUTPUT_DIR/'customer_lifecycle';PARTITION_DIR.mkdir(parents=True,exist_ok=True)
STATES=['New','Active','At-risk','Churned','Win-back']
STATE_CODE={name:i for i,name in enumerate(STATES)}
REASONS=['new_bank_relationship','observed_presence','short_panel_absence','panel_absence_proxy','return_after_churn_proxy']
HORIZONS = list(range(1, 13))
plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
def mtext(m):return f'{(int(m)-1)//12}-{(int(m)-1)%12+1:02d}'
def savefig(name):plt.gcf().savefig(OUTPUT_DIR/(name+'.png'),bbox_inches='tight');plt.show();plt.close()
assert SAMPLE_MOD>=1 and CHURN_GAP>=1 and NEW_MONTHS>=1


## 1. Input
File monthly RFM phải đủ tháng liên tiếp; legacy chỉ dùng is_observed=1. Không lấy dòng stale làm snapshot thật.

Metadata tuỳ chọn (`train_ver2.csv` hoặc bảng chuẩn) chỉ đọc ID, tháng và fecha_alta. Notebook không đọc hoặc sử dụng ind_actividad_cliente/is_active. Thiếu ngày mở quan hệ không chặn pipeline: chưa gán New, vẫn gán Active/Win-back cho khách có snapshot. Không dùng lần đầu xuất hiện trong dataset thay ngày mở quan hệ.


In [ ]:
root=Path(INPUT_DIR); prefix='rfm_' if INPUT_FORMAT=='rfm' else 'customer_features_'
rx=re.compile(r'^'+prefix+r'(\d{4})-(\d{2})\.parquet$')
files={}
for p in root.rglob(prefix+'*.parquet'):
    z=rx.match(p.name)
    if not z:continue
    m=int(z[1])*12+int(z[2]);assert m not in files,f'Trùng tháng {mtext(m)}';files[m]=p
months=np.array(sorted(files),dtype=int)
assert len(months)>1 and np.diff(months).tolist()==[1]*(len(months)-1),'Thiếu file tháng / chưa tìm đúng INPUT_DIR'
# First pass: only observed IDs, no full history frame in RAM.
all_ids=[];inventory=[]
for m in months:
    names=pq.ParquetFile(files[m]).schema_arrow.names
    assert 'customer_id' in names
    cols=['customer_id']+(['is_observed'] if INPUT_FORMAT=='legacy' else [])
    q=pd.read_parquet(files[m],columns=cols)
    assert q.customer_id.notna().all() and q.customer_id.is_unique
    if INPUT_FORMAT=='legacy':
        assert q.is_observed.isin([0,1]).all();q=q.loc[q.is_observed.eq(1)]
    inventory.append({'month':mtext(m),'observed_all_n':len(q)})
    all_ids.append(q.loc[q.customer_id.mod(SAMPLE_MOD).eq(0),'customer_id'].to_numpy())
ids=pd.Index(np.unique(np.concatenate(all_ids)),name='customer_id');del all_ids
assert len(ids)
metadata_parts={}
if METADATA_PATH is None and Path('/kaggle/input').exists():
    raw_candidates=list(Path('/kaggle/input').rglob('train_ver2.csv'))
    if len(raw_candidates)==1:
        METADATA_PATH=str(raw_candidates[0]);print('Tự tìm thấy dữ liệu gốc để bổ sung metadata:',METADATA_PATH)
    elif len(raw_candidates)>1:print('Có nhiều train_ver2.csv: điền METADATA_PATH để chọn nguồn.')
if METADATA_PATH:
    meta_path=Path(METADATA_PATH)
    aliases={'ncodpers':'customer_id','fecha_dato':'record_date'}
    source_names=list(pq.ParquetFile(meta_path).schema_arrow.names) if meta_path.suffix=='.parquet' else list(pd.read_csv(meta_path,nrows=0).columns)
    selected=[c for c in source_names if c in ['customer_id','ncodpers','record_date','fecha_dato','month_idx','fecha_alta']]
    cache=OUTPUT_DIR/'metadata_cache'/uuid.uuid4().hex;cache.mkdir(parents=True)
    iterator=[pd.read_parquet(meta_path,columns=selected)] if meta_path.suffix=='.parquet' else pd.read_csv(meta_path,usecols=selected,chunksize=CSV_CHUNKSIZE,low_memory=False)
    for chunk_i,q in enumerate(iterator):
        q=q.rename(columns=aliases)
        assert 'customer_id' in q and ('record_date' in q or 'month_idx' in q)
        q['customer_id']=pd.to_numeric(q.customer_id,errors='raise')
        q=q.loc[q.customer_id.isin(ids)].copy()
        if 'month_idx' not in q:
            dates=pd.to_datetime(q.record_date,errors='raise');q['month_idx']=dates.dt.year*12+dates.dt.month
        for m,g in q.loc[q.month_idx.isin(months)].groupby('month_idx'):
            dest=cache/f'meta_{int(m)}_{chunk_i}.parquet';g.to_parquet(dest,index=False)
            metadata_parts.setdefault(int(m),[]).append(dest)
    print('Metadata đã chia theo tháng cho phiên chạy.')
def read_current(m):
    names=pq.ParquetFile(files[m]).schema_arrow.names
    cols=[c for c in ['customer_id','is_observed','fecha_alta',
        'F_events','F_event_months','R','M_group_points','products_complete','portfolio_has_unknown','portfolio_code'] if c in names]
    q=pd.read_parquet(files[m],columns=cols)
    if INPUT_FORMAT=='legacy':q=q.loc[q.is_observed.eq(1)].copy()
    q=q.loc[q.customer_id.isin(ids)].copy()
    if 'products_complete' not in q:q['products_complete']=~q.portfolio_has_unknown.eq(True) if 'portfolio_has_unknown' in q else False
    if m in metadata_parts:
        meta=pd.concat([pd.read_parquet(p) for p in metadata_parts[m]],ignore_index=True)
        assert not meta.duplicated('customer_id').any(),f'Trùng metadata {mtext(m)}'
        use=[c for c in ['fecha_alta'] if c in meta]
        q=q.merge(meta[['customer_id']+use],on='customer_id',how='left',validate='one_to_one',suffixes=('','_meta'))
        for c in use:
            if c+'_meta' in q:
                q[c]=pd.to_datetime(q[c+'_meta'],errors='coerce').fillna(pd.to_datetime(q[c],errors='coerce'))
                q=q.drop(columns=c+'_meta')
    if 'fecha_alta' not in q:q['fecha_alta']=pd.NaT
    q['fecha_alta']=pd.to_datetime(q.fecha_alta,errors='coerce')
    assert q.customer_id.is_unique
    return q.set_index('customer_id')
pd.DataFrame(inventory).to_csv(OUTPUT_DIR/'inventory.csv',index=False)
print(f'{len(months)} tháng, {len(ids):,} khách; SAMPLE_MOD={SAMPLE_MOD}')

# Auto locate assignments if exactly one method folder is present; otherwise select explicitly.
if RFM_ASSIGNMENT_DIR is None:
    search_root=Path('/kaggle/input') if Path('/kaggle/input').exists() else root
    found={p.parent for p in search_root.rglob(f'assignments_{RFM_METHOD}_*.parquet')}
    if len(found)==1:RFM_ASSIGNMENT_DIR=str(next(iter(found)));print('Assignments RFM:',RFM_ASSIGNMENT_DIR)
    else:print(f'Có {len(found)} thư mục assignments phù hợp. Điền RFM_ASSIGNMENT_DIR để chạy phần kết hợp.')


## 2. Rule chuyển trạng thái — thông tin có tại tháng t
| Trạng thái | Điều kiện |
|---|---|
| Churned proxy | Vắng liên tiếp ≥6 tháng |
| At-risk | Vắng liên tiếp 1–5 tháng |
| Win-back | Snapshot đầu quay lại sau khoảng vắng ≥6 tháng; chỉ tháng quay lại |
| New | Có snapshot, mở quan hệ ngân hàng trong tháng hiện tại hoặc tháng trước |
| Active | Có snapshot, không thuộc Win-back hoặc New |

Khi có snapshot: **Win-back → New → Active** theo ưu tiên. Khách quay lại sau khoảng vắng ngắn hơn 6 tháng trở về Active (hoặc New nếu có ngày mở quan hệ phù hợp).

Không có Unknown/Inactive. Không dùng cờ hoạt động, M giảm hay giảm sản phẩm để chuyển lifecycle. Giảm/đổi danh mục vẫn được xuất để phân tích. Active mô tả sự hiện diện, Churned là proxy mất quan sát.


In [ ]:
N=len(ids);T=len(months)
# Small integer matrices for downstream evaluation, not a full object customer-month frame.
observed=np.zeros((N,T),bool);life=np.full((N,T),255,np.uint8) # 255=chưa từng thấy, không xuất thành lifecycle
acq_valid=np.zeros((N,T),bool);acq_positive=np.zeros((N,T),bool)
known=np.zeros(N,bool);gap=np.zeros(N,np.int16);first_seen=np.full(N,-1,int)
prev_complete=np.zeros(N,bool);prev_f=np.full(N,np.nan)
last_code=np.zeros(N,np.uint32);last_code_valid=np.zeros(N,bool)
counts_rows=[];transitions=[];coverage_rows=[];return_parts=[]
last_available_rfm=np.full(N,None,dtype=object);last_available_rfm_month=np.full(N,-1,dtype=int)
def popcount(a):
    result=np.zeros(len(a),np.int16)
    for bit in range(24):result+=((a>>bit)&1).astype(np.int16)
    return result
for j,m in enumerate(months):
    q=read_current(int(m));pos=ids.get_indexer(q.index);obs=np.zeros(N,bool);obs[pos]=True
    observed[:,j]=obs;prev_obs=observed[:,j-1] if j else np.zeros(N,bool)
    prev_gap=gap.copy();was_known=known.copy();known|=obs
    first_seen[(first_seen<0)&obs]=m
    gap=np.where(obs,0,np.where(known,gap+1,0)).astype(np.int16)
    complete=np.zeros(N,bool);complete[pos]=q.products_complete.eq(True).to_numpy()
    f=np.full(N,np.nan)
    if 'F_events' in q:
        assert q.F_events.notna().all() and np.isfinite(q.F_events).all() and q.F_events.ge(0).all() and q.F_events.mod(1).eq(0).all()
        f[pos]=q.F_events.to_numpy(float)
    delta=f-prev_f;pair=prev_obs&obs&complete&prev_complete&np.isfinite(delta)
    assert not (pair&(delta<0)).any(),'F_events giảm trong cặp hợp lệ'
    acq_valid[:,j]=pair;acq_positive[:,j]=pair&(delta>0)
    returned=obs&was_known&~prev_obs;winback=returned&(prev_gap>=CHURN_GAP)
    bank_age=np.full(N,np.nan)
    bank_month=q.fecha_alta.dt.year*12+q.fecha_alta.dt.month
    bank_age[pos]=m-bank_month.to_numpy(float)
    new_bank=obs&np.isfinite(bank_age)&(bank_age>=0)&(bank_age<NEW_MONTHS)
    # Only current presence, absence duration and current bank-opening date define lifecycle.
    state=np.full(N,255,np.uint8)
    state[obs]=STATE_CODE['Active']
    state[new_bank]=STATE_CODE['New']
    state[winback]=STATE_CODE['Win-back'] # overrides New/Active only in return month
    absent=known&~obs
    state[absent]=STATE_CODE['At-risk']
    churn=absent&(gap>=CHURN_GAP)
    state[churn]=STATE_CODE['Churned']
    assert (state[known]<len(STATES)).all(), 'Known customer has no valid state'
    reason=state.copy() # aligned with REASONS in cell 2
    life[:,j]=state
    # Portfolio change vs preceding month, or last observed month on a return.
    cur_code=np.zeros(N,np.uint32);cur_code_valid=np.zeros(N,bool)
    if 'portfolio_code' in q:
        codes=pd.to_numeric(q.portfolio_code,errors='coerce').to_numpy(float,na_value=np.nan)
        finite=np.isfinite(codes)
        assert (codes[finite]>=0).all() and (codes[finite]<2**24).all() and (codes[finite]==np.floor(codes[finite])).all()
        cur_code[pos]=np.where(finite,codes,0).astype(np.uint32)
        cur_code_valid[pos]=finite&complete[pos]
    comparable=obs&last_code_valid&cur_code_valid
    added=popcount(cur_code&~last_code);removed=popcount(last_code&~cur_code)
    change=np.full(N,'NotComparable',object)
    change[comparable&(added==0)&(removed==0)]='Maintain'
    change[comparable&(added>0)&(removed==0)]='Increase'
    change[comparable&(added==0)&(removed>0)]='Decrease'
    change[comparable&(added>0)&(removed>0)]='Mixed'
    k=np.flatnonzero(known)
    out=pd.DataFrame({'customer_id':ids[k],'month':mtext(m),'month_idx':m,
        'is_observed':obs[k],'missing_months':gap[k],
        'first_seen_month_idx':first_seen[k],'is_left_censored':first_seen[k]==months[0],
        'bank_relationship_age_months':bank_age[k],
        'lifecycle':np.array(STATES)[state[k]],'lifecycle_reason':np.array(REASONS)[reason[k]],
        'returned_to_panel':returned[k],'absence_before_return':np.where(returned[k],prev_gap[k],np.nan),
        'is_new_to_panel':obs[k]&(first_seen[k]==m)&(m!=months[0]),
        'acquisition_pair_valid':pair[k],
        'has_acquisition_month':pd.array(np.where(pair[k],acq_positive[k,j],np.nan),dtype='boolean'),
        'portfolio_comparable':comparable[k],'portfolio_change':change[k],
        'added_products':np.where(comparable[k],added[k],np.nan),
        'removed_products':np.where(comparable[k],removed[k],np.nan),
        'portfolio_comparison':'last_observed_snapshot','rule_version':RULE_VERSION})
    # Current RFM only for observed rows, never carry stale R/F/M into current-month segmentation.
    feature_cols=[c for c in ['R','F_event_months','F_events','M_group_points'] if c in q]
    out=out.merge(q[feature_cols],left_on='customer_id',right_index=True,how='left',validate='one_to_one')
    if RFM_ASSIGNMENT_DIR:
        path=Path(RFM_ASSIGNMENT_DIR)/f'assignments_{RFM_METHOD}_{mtext(m)}.parquet'
        if path.exists():
            a=pd.read_parquet(path)
            if 'method' in a:a=a.loc[a.method.eq(RFM_METHOD)]
            assert a.customer_id.is_unique
            date_col='as_of_month' if 'as_of_month' in a else ('month' if 'month' in a else None)
            if date_col:
                dates=pd.to_datetime(a[date_col].astype(str),errors='raise')
                assert (dates.dt.year*12+dates.dt.month).eq(m).all(), 'Assignment month mismatch'
            out=out.merge(a[['customer_id','cluster']].rename(columns={'cluster':'rfm_cluster'}),on='customer_id',how='left',validate='one_to_one')
            out.loc[~out.is_observed,'rfm_cluster']=pd.NA
        else:out['rfm_cluster']=pd.NA
    if 'rfm_cluster' not in out:out['rfm_cluster']=pd.NA
    out['rfm_status']=np.where(~out.is_observed,'NoCurrentSnapshot',
        np.where(out.F_events.eq(0),'NoObservedEvent',
        np.where(out.rfm_cluster.notna(),'Assigned','MissingAssignment'))) if 'F_events' in out else np.where(out.is_observed,'MissingAssignment','NoCurrentSnapshot')
    out.loc[~out.rfm_status.eq('Assigned'),'rfm_cluster']=pd.NA
    out['rfm_context']=out.rfm_cluster.astype('string').fillna(out.rfm_status.astype('string'))
    out['combined_segment']=out.rfm_context+' | '+out.lifecycle.astype('string')
    assigned=out.rfm_status.eq('Assigned').to_numpy();assigned_pos=k[assigned]
    last_available_rfm[assigned_pos]=out.loc[assigned,'rfm_cluster'].astype(str).to_numpy()
    last_available_rfm_month[assigned_pos]=m
    out['last_available_rfm_cluster']=pd.array(last_available_rfm[k],dtype='string')
    out['last_available_rfm_month_idx']=np.where(last_available_rfm_month[k]>=0,last_available_rfm_month[k],np.nan)
    out['last_available_rfm_age_months']=np.where(last_available_rfm_month[k]>=0,m-last_available_rfm_month[k],np.nan)
    assert out.customer_id.is_unique and len(out)==int(known.sum())
    assert out.loc[out.lifecycle.eq('Churned'),'missing_months'].ge(CHURN_GAP).all()
    assert out.loc[out.lifecycle.eq('Win-back'),'absence_before_return'].ge(CHURN_GAP).all()
    out.to_parquet(PARTITION_DIR/f'lifecycle_{mtext(m)}.parquet',index=False)
    for state_name,g in out.groupby('lifecycle'):
        counts_rows.append({'month':mtext(m),'lifecycle':state_name,'n':len(g)})
    coverage_rows.append({'month':mtext(m),'observed_n':int(obs.sum()),
        'bank_date_known_n':int((obs&np.isfinite(bank_age)&(bank_age>=0)).sum()),
        'missing_bank_date_n':int((obs&~(np.isfinite(bank_age)&(bank_age>=0))).sum()),
        'return_n':int(returned.sum()),'winback_n':int(winback.sum())})
    if returned.any():return_parts.append(out.loc[out.returned_to_panel,['customer_id','month','lifecycle','absence_before_return','portfolio_comparable','portfolio_change','added_products','removed_products']])
    if j:
        valid=was_known
        t=pd.crosstab(np.array(STATES)[life[valid,j-1]],np.array(STATES)[state[valid]])
        for a in t.index:
            for b in t.columns:
                if t.loc[a,b]:transitions.append({'source_month':mtext(months[j-1]),'target_month':mtext(m),'from_state':a,'to_state':b,'n':int(t.loc[a,b])})
    # Save last real portfolio, including invalid flag: never bridge to an earlier complete snapshot.
    last_code[obs]=cur_code[obs];last_code_valid[obs]=cur_code_valid[obs]
    prev_complete=complete;prev_f=f
counts=pd.DataFrame(counts_rows);coverage=pd.DataFrame(coverage_rows)
transition_counts=pd.DataFrame(transitions,columns=['source_month','target_month','from_state','to_state','n'])
returns=pd.concat(return_parts,ignore_index=True) if return_parts else pd.DataFrame()
for name,frame in [('state_counts',counts),('coverage',coverage),('transitions',transition_counts),('return_portfolios',returns)]:frame.to_csv(OUTPUT_DIR/(name+'.csv'),index=False)
print('Đã gán và xuất từng tháng tại',PARTITION_DIR)


## 3. Số khách, ngày mở quan hệ và chuyển trạng thái
Ngày mở quan hệ chỉ ảnh hưởng việc nhận diện New. Ma trận đo trạng thái tháng sau theo trạng thái tháng nguồn, chỉ trên khách đã được quan sát; không đưa khách chưa từng xuất hiện vào một trạng thái giả. Không dùng độ bao phủ activity nữa.


In [ ]:
coverage['bank_date_coverage']=coverage.bank_date_known_n/coverage.observed_n
view=coverage[['month','observed_n','bank_date_coverage','missing_bank_date_n','winback_n']].copy()
view['bank_date_coverage']=view.bank_date_coverage.map(lambda x:f'{x:.1%}')
display(view)
if coverage.bank_date_known_n.sum()==0:
    print('Không có fecha_alta: chưa xác nhận New; khách có snapshot vẫn là Active hoặc Win-back. METADATA_PATH chỉ cần nếu muốn nhận diện New.')
monthly=counts.pivot(index='month',columns='lifecycle',values='n').fillna(0).reindex(columns=STATES,fill_value=0)
display(monthly.tail(6))
fig,ax=plt.subplots(figsize=(11,4))
for s in ['At-risk','Churned','Win-back']:ax.plot(monthly.index,monthly[s],marker='o',label=s)
ax.set(title='At-risk / Churn proxy / Win-back theo tháng',ylabel='Khách');ax.tick_params(axis='x',rotation=45);ax.legend()
plt.tight_layout();savefig('state_counts')
if not transition_counts.empty:
    matrix=transition_counts.pivot_table(index='from_state',columns='to_state',values='n',aggfunc='sum',fill_value=0).reindex(index=STATES,columns=STATES,fill_value=0)
    rates=matrix.div(matrix.sum(axis=1).replace(0,np.nan),axis=0)
    rates.to_csv(OUTPUT_DIR/'transition_rates.csv');display(rates.map(lambda x:'NA' if pd.isna(x) else f'{x:.1%}'))


## 4. Mỗi trạng thái có giữ ý nghĩa trong tương lai không? Đúng mốc +1…+6
Chọn trạng thái ở t, đo snapshot đúng t+h. Không cộng các tháng trước; mốc vượt dataset để thiếu. Acquisition t+1 chỉ trên cặp portfolio liên tiếp hợp lệ; khách vắng tại t không bị tính là không acquisition. Bảng này mô tả theo tháng nguồn, nên kiểm tra batch và số mẫu trước khi kết luận rule tốt.


In [ ]:
eval_rows=[];acq_rows=[]
for j,m in enumerate(months):
    known_at=first_seen<=m # first_seen filled by past-only pass; filter out future entrants at source
    for si,s in enumerate(STATES):
        mask=known_at&(life[:,j]==si);denom=int(mask.sum())
        if not denom:continue
        for h in HORIZONS:
            enough=j+h<T
            present=int(observed[mask,j+h].sum()) if enough else np.nan
            eval_rows.append({'source_month':mtext(m),'lifecycle':s,'horizon':h,'n':denom,
                'followup_available':enough,'present_n':present,'present_rate':present/denom if enough else np.nan})
        if j+1<T:
            valid=mask&acq_valid[:,j+1]
            acq_rows.append({'month':mtext(m),'lifecycle':s,'n':denom,'label_n':int(valid.sum()),
                'positive_n':int(acq_positive[valid,j+1].sum())})
evaluation=pd.DataFrame(eval_rows);acq_eval=pd.DataFrame(acq_rows)
evaluation.to_csv(OUTPUT_DIR/'exact_presence_by_state_month.csv',index=False)
valid_eval=evaluation.loc[evaluation.followup_available]
summary=valid_eval.groupby(['lifecycle','horizon'],as_index=False)[['n','present_n']].sum()
summary['present_rate']=summary.present_n/summary.n
summary.to_csv(OUTPUT_DIR/'exact_presence_summary.csv',index=False)
table=summary.pivot(index='lifecycle',columns='horizon',values='present_rate').reindex(STATES)
display(table.map(lambda x:'NA' if pd.isna(x) else f'{x:.2%}'))
acq_summary=acq_eval.groupby('lifecycle',as_index=False)[['n','label_n','positive_n']].sum()
acq_summary['label_coverage']=acq_summary.label_n/acq_summary.n
acq_summary['acquisition_rate']=acq_summary.positive_n/acq_summary.label_n.replace(0,np.nan)
acq_eval.to_csv(OUTPUT_DIR/'acquisition_by_state_month.csv',index=False);acq_summary.to_csv(OUTPUT_DIR/'acquisition_summary.csv',index=False)
display(acq_summary)
if not returns.empty:
    print('Danh mục khi quay lại: số lượt, chỉ trên cặp hợp lệ')
    r=returns.loc[returns.portfolio_comparable]
    display(pd.crosstab(r.lifecycle,r.portfolio_change))
    print(f'Cặp danh mục hợp lệ: {len(r):,}/{len(returns):,}; xem thêm return_portfolios.csv để audit tháng và khoảng vắng.')


## 5. Bàn giao và cách dùng với RFM
Output `customer_lifecycle/lifecycle_YYYY-MM.parquet`: một dòng/khách đã được thấy/tháng, kể cả tháng vắng. Có trạng thái, lý do, version, gap, cờ hiện diện, event hợp lệ và thay đổi danh mục. Feature R/F/M chỉ tồn tại khi có snapshot tại tháng t; không carry dữ liệu stale.

Ghép assignments Hybrid bằng customer_id + tháng, giữ lifecycle và rfm_cluster là hai cột độc lập. Missing RFM có thể do NoObservedEvent hoặc chưa có assignments ở tháng đó, không phải churn. Nếu muốn đo hiệu quả bổ sung của lifecycle, cần ablation ngoài mẫu RFM vs RFM+lifecycle; các bảng mô tả hiện tại chưa chứng minh cải thiện dự báo.

Kiểm tra cuối: độ bao phủ ngày mở quan hệ, batch 07/2015, độ ổn định theo tháng, số label hợp lệ từng state. Rule Churned vẫn là proxy mất quan sát. Ngưỡng 6 đang được áp dụng theo lựa chọn, chưa tuyên bố tối ưu. Không gán Inactive.

**Phần kết hợp bên dưới:** từng partition có `rfm_cluster`, `rfm_status`, `combined_segment`. `NoObservedEvent` vẫn nằm trong dữ liệu mô tả nhưng được tách khỏi benchmark HasEvent. Khách vắng không được gán bucket RFM cũ như thể đó là trạng thái hiện tại.


In [ ]:
spec={'rule_version':RULE_VERSION,'churn_gap':CHURN_GAP,'new_months':NEW_MONTHS,
    'sample_mod':SAMPLE_MOD,'months':[mtext(m) for m in months],'states':STATES,
    'priority':['Churned/At-risk absence','Win-back','New bank relationship','Active presence'],
    'as_of_rule':'only source/current and past snapshots for state; future used only for evaluation',
    'evaluation_horizons':HORIZONS,'return_definition':'observed exactly at t+h',
    'winback_duration':'return month only; next month reevaluate',
    'churn_scope':'panel absence proxy, not confirmed bank churn',
    'active_scope':'observed snapshot, not transaction activity','activity_flag_used':False,
    'missing_bank_date_behavior':'observed -> Active unless Win-back; New unconfirmed',
    'new_scope':'fecha_alta current snapshot; no New inferred solely from panel entry',
    'rfm_method':RFM_METHOD if RFM_ASSIGNMENT_DIR else None,'rfm_fit_month':RFM_EVAL_FIT_MONTH,
    'combined_label':'rfm_context | lifecycle','combined_benchmark':'rolling historical group-rate, common matched acquisition cohort',
    'refresh':'monthly','partition_key':['customer_id','month_idx']}
(OUTPUT_DIR/'lifecycle_feature_spec.json').write_text(json.dumps(spec,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'manifest.json').write_text(json.dumps({'partitions':[f'customer_lifecycle/lifecycle_{mtext(m)}.parquet' for m in months],
    'rule_version':RULE_VERSION,'customers':N,'sample_mod':SAMPLE_MOD},indent=2),encoding='utf-8')
print('Hoàn tất: partitions, transition matrix, evaluations, coverage, feature spec và manifest tại',OUTPUT_DIR)


## 6. Profile kết hợp: RFM mô tả hành vi, lifecycle mô tả trạng thái
**Ví dụ:** `R1_F3_M5 | Active` = bucket RFM này và đang hiện diện; `last_available_rfm_cluster = R1_F3_M5`, lifecycle = At-risk: bucket lịch sử trước khi vắng, có tháng chấm kèm theo; không phải RFM hiện tại. Khách không xuất hiện tháng t có nhãn `NoCurrentSnapshot | Churned`, không carry bucket cũ.

Bảng chính: số khách từng lifecycle × bucket tại tháng cuối có assignments; bảng coverage cho biết bao nhiêu khách được ghép. Không gọi MissingAssignment là NoEvent hoặc churn.

Phần acquisition chỉ đánh giá trên khách có assignment và label cặp t→t+1 hợp lệ. Vì khách vắng không có cặp này, benchmark acquisition chưa đánh giá chất lượng dự báo return/churn. NoObservedEvent được giữ riêng trong output, không trộn với benchmark HasEvent.

**Khách đang vắng:** vẫn giữ `last_available_rfm_cluster` kèm tháng chấm và tuổi thông tin để xem họ thuộc bucket nào lần gần nhất có assignment. Đây là thông tin lịch sử, không phải bucket hiện tại; benchmark chỉ dùng `rfm_cluster` hiện tại.


In [ ]:
joint_rows=[];join_coverage=[];joint_outcomes=[]
for j,m in enumerate(months):
    frame=pd.read_parquet(PARTITION_DIR/f'lifecycle_{mtext(m)}.parquet')
    join_coverage.append({'month':mtext(m),'observed_n':int(frame.is_observed.sum()),
        'assigned_n':int(frame.rfm_status.eq('Assigned').sum()),
        'no_event_n':int(frame.rfm_status.eq('NoObservedEvent').sum()),
        'missing_assignment_n':int(frame.rfm_status.eq('MissingAssignment').sum())})
    for (rfm,lc),g in frame.groupby(['rfm_context','lifecycle']):
        joint_rows.append({'month':mtext(m),'rfm_context':rfm,'lifecycle':lc,'n':len(g)})
    if j+1<T:
        pos=ids.get_indexer(frame.customer_id);valid=frame.rfm_status.eq('Assigned').to_numpy()&acq_valid[pos,j+1]
        v=frame.loc[valid,['customer_id','rfm_cluster','lifecycle','combined_segment']].copy()
        v['y']=acq_positive[pos[valid],j+1].astype(int)
        if len(v):
            g=v.groupby(['rfm_cluster','lifecycle'],as_index=False).y.agg(n='size',events='sum',rate='mean')
            joint_outcomes.append(g.assign(month=mtext(m)))
join_coverage=pd.DataFrame(join_coverage)
join_coverage['assigned_share_observed']=join_coverage.assigned_n/join_coverage.observed_n
joint_profiles=pd.DataFrame(joint_rows)
joint_profiles.to_csv(OUTPUT_DIR/'rfm_lifecycle_profiles.csv',index=False)
join_coverage.to_csv(OUTPUT_DIR/'rfm_join_coverage.csv',index=False);display(join_coverage)
if joint_outcomes:
    joint_outcomes=pd.concat(joint_outcomes,ignore_index=True)
    joint_outcomes.to_csv(OUTPUT_DIR/'rfm_lifecycle_acquisition_by_month.csv',index=False)
    shown_month=join_coverage.loc[join_coverage.assigned_n.gt(0),'month'].max()
    shown=joint_profiles.loc[joint_profiles.month.eq(shown_month)]
    buckets=shown.groupby('rfm_context').n.sum().nlargest(12).index
    display(shown.loc[shown.rfm_context.isin(buckets)].pivot(index='rfm_context',columns='lifecycle',values='n').fillna(0))
    print('Profile tại',shown_month,'; chỉ hiện 12 nhóm đông nhất. CSV giữ toàn bộ.')
else:print('Chưa có assignments + label hợp lệ: chưa lập bảng acquisition kết hợp. Điền RFM_ASSIGNMENT_DIR.')


## 7. Lifecycle có bổ sung thông tin cho RFM không? So sánh ngoài mẫu theo thời gian
So **RFM-only**, **Lifecycle-only**, **RFM+Lifecycle** trên cùng khách HasEvent có assignment và label acquisition hợp lệ. Khởi tạo tỷ lệ nhóm từ tháng fit (mặc định 11/2015), chấm các tháng sau trước khi cập nhật label mới.

Score nhóm là tỷ lệ acquisition lịch sử, regularize 100 pseudo-observations. Nhóm kết hợp chưa có lịch sử backoff về score RFM. Đây là benchmark group-rate, chưa phải logistic ablation liên tục hoặc so sánh lại mọi phương pháp phân cụm.

**Đọc:** AP/Lift10 cao hơn, Brier thấp hơn là tốt. Báo delta từng tháng; không tự gọi thắng chỉ vì trung bình tăng nhẹ. Điểm lịch sử chỉ dùng label đã biết đến thời điểm chấm. Lift10 xử lý ties bằng chọn tỷ lệ kỳ vọng, không dựa vào thứ tự ID.


In [ ]:
from sklearn.metrics import average_precision_score,roc_auc_score,brier_score_loss
fit_period=pd.Period(RFM_EVAL_FIT_MONTH,freq='M');fit_idx=fit_period.year*12+fit_period.month
assert fit_idx in months,'Tháng fit không có trong dữ liệu'
assert GROUP_RATE_PRIOR_N>0
hist={name:pd.DataFrame(columns=['n','events'],dtype=float) for name in ['RFM','Lifecycle','RFM+Lifecycle']}
bench_rows=[];n_train=0;events_train=0

def score_groups(keys,history,fallback):
    n=keys.map(history.n).fillna(0).to_numpy(float)
    k=keys.map(history.events).fillna(0).to_numpy(float)
    fallback=np.broadcast_to(np.asarray(fallback,dtype=float),n.shape)
    return (k+GROUP_RATE_PRIOR_N*fallback)/(n+GROUP_RATE_PRIOR_N)
def lift10(y,score):
    n=len(y);quota=max(1,int(np.ceil(.1*n)))
    threshold=np.sort(score)[-quota];high=score>threshold;ties=score==threshold
    needed=quota-int(high.sum());positive=float(y[high].sum())+needed*float(y[ties].mean())
    return (positive/quota)/float(y.mean()) if y.mean()>0 else np.nan
for j,m in enumerate(months[:-1]):
    if m<fit_idx:continue
    frame=pd.read_parquet(PARTITION_DIR/f'lifecycle_{mtext(m)}.parquet')
    pos=ids.get_indexer(frame.customer_id)
    valid=frame.rfm_status.eq('Assigned').to_numpy()&acq_valid[pos,j+1]
    v=frame.loc[valid,['rfm_cluster','lifecycle','combined_segment']].copy().reset_index(drop=True)
    y=acq_positive[pos[valid],j+1].astype(int)
    if not len(v):continue
    keys={'RFM':v.rfm_cluster.astype(str),'Lifecycle':v.lifecycle.astype(str),'RFM+Lifecycle':v.combined_segment.astype(str)}
    if m>fit_idx and n_train>0:
        global_rate=events_train/n_train
        scores={'RFM':score_groups(keys['RFM'],hist['RFM'],global_rate),
            'Lifecycle':score_groups(keys['Lifecycle'],hist['Lifecycle'],global_rate)}
        scores['RFM+Lifecycle']=score_groups(keys['RFM+Lifecycle'],hist['RFM+Lifecycle'],scores['RFM'])
        for name,score in scores.items():
            ap=average_precision_score(y,score) if y.sum()>0 else np.nan
            auc=roc_auc_score(y,score) if np.unique(y).size==2 else np.nan
            bench_rows.append({'month':mtext(m),'method':name,'n':len(y),'positives':int(y.sum()),
                'AP':ap,'AUC':auc,'Lift10':lift10(y,score),'Brier':brier_score_loss(y,score)})
    # Result of source t becomes available at t+1, after scoring source t.
    for name,key in keys.items():
        counts_new=pd.DataFrame({'key':key,'y':y}).groupby('key').y.agg(n='size',events='sum')
        hist[name]=hist[name].add(counts_new,fill_value=0)
    n_train+=len(y);events_train+=int(y.sum())
benchmark=pd.DataFrame(bench_rows)
if benchmark.empty:print('Chưa đủ fit/history/eval cohort. Không kết luận kết hợp tốt hơn; kiểm tra assignments và fit month.')
else:
    assert benchmark.groupby('month').n.nunique().eq(1).all()
    benchmark.to_csv(OUTPUT_DIR/'rfm_lifecycle_benchmark_monthly.csv',index=False)
    result=benchmark.groupby('method')[['AP','AUC','Lift10','Brier']].mean()
    display(result.round(4));result.to_csv(OUTPUT_DIR/'rfm_lifecycle_benchmark_summary.csv')
    base=benchmark.loc[benchmark.method.eq('RFM')].set_index('month')
    joint=benchmark.loc[benchmark.method.eq('RFM+Lifecycle')].set_index('month')
    delta=(joint[['AP','Lift10','Brier']]-base[['AP','Lift10','Brier']]).add_prefix('delta_')
    display(delta.round(4));delta.to_csv(OUTPUT_DIR/'rfm_lifecycle_paired_delta.csv')
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for ax,metric in zip(axes,['AP','Lift10']):
        for name,g in benchmark.groupby('method'):ax.plot(g.month,g[metric],marker='o',label=name)
        ax.set_title(metric+' theo tháng nguồn — cùng cohort');ax.tick_params(axis='x',rotation=45)
    axes[-1].legend();plt.tight_layout();savefig('rfm_lifecycle_comparison')
print('Không có ablation return/churn trong benchmark acquisition này; dùng exact_presence_by_state_month.csv để mô tả return.')


## Tóm tắt sử dụng
- **Phân tích:** dùng RFM × lifecycle để giải thích “khách thuộc bucket nào, đang ở trạng thái nào”. Không ép thành một hệ phân cụm mới.
- **Hành động:** Active + RFM ưu tiên cao là ứng viên cross-sell; At-risk là ứng viên kiểm tra suy giảm; Win-back cần xem danh mục khi quay lại. Churn proxy không có RFM hiện tại.
- **Kiểm chứng:** chỉ nói lifecycle bổ sung khả năng dự báo nếu benchmark ngoài mẫu cải thiện đủ nhất quán; nếu phần lớn matched khách đều Active thì có thể ít thông tin mới.
- **Giới hạn:** RFM assignments hiện thường chỉ có 11/2015–04/2016 và HasEvent; không có bucket không đồng nghĩa churn. Ngày mở quan hệ thiếu khiến New chưa xác nhận được. Kiểm tra coverage trước khi dùng kết quả.

**Lưu ý với rule hiện diện:** benchmark acquisition yêu cầu snapshot t và t+1, nên At-risk/Churned (vắng tại t) không nằm trong cohort này. Benchmark chủ yếu so Active/New/Win-back. Chỉ có Active thì lifecycle không bổ sung khả năng phân biệt trên cohort acquisition; đây là kết quả hợp lý, không phải lỗi ghép.
